In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from glob import glob
from autopath.ap_PLIP import ProteinLigandAnalyzer

# LIE

In [ ]:
allsystems = glob('*/lie/LIE_ALL.csv')
alldfs = []
for pathfname in allsystems:
    sysname = pathfname.split('/')[0]
    df = pd.read_csv(pathfname, index_col=0)
    df['system'] = sysname
    alldfs.append(df)
df_all = pd.concat(alldfs)
df_all.head()

In [ ]:
for component in ['Total', 'VDW', 'EELEC']:
    plt.figure(figsize=(4, 6))
    for sysname, df_sys in df_all.groupby('system'):
        mean_dG = df_sys[component].mean()
        std_dG = df_sys[component].std()
        plt.errorbar(mean_dG, sysname, xerr=std_dG, fmt='o', label=sysname)
        # plt.bar(mean_dG, sysname, yerr=std_dG, capsize=5)
        plt.title(f'LIE {component}')
        plt.xlabel('Energy (kcal/mol)'); plt.ylabel('System')
    plt.savefig(f'lie_all_{component}.png', dpi=300)
    plt.show()
    plt.close()
    
    plt.figure(figsize=(4, 6))
    mean_std_dG = df_all.groupby('system')[component].agg(['mean', 'std']).sort_values(by='mean', ascending=True)
    systems = mean_std_dG.index
    means = mean_std_dG['mean']
    stds = mean_std_dG['std']
    
    # Assign different colors to each bar
    colors = sns.color_palette("coolwarm", len(systems))
    
    plt.barh(systems, means, xerr=stds, capsize=5, edgecolor='black', color=colors)
    plt.title(f'LIE {component}')
    plt.xlabel('Energy (kcal/mol)')
    plt.ylabel('System')
    plt.gca().invert_yaxis()  # Invert y-axis to have the highest bar at the top
    plt.savefig(f'lie_all_{component}_sorted_colored.png', dpi=300)
    plt.show()
    plt.close()
        
    plt.figure(figsize=(6, 4))
    std_dG_by_system = df_all.groupby('system')[component].std().sort_values(ascending=True)
    systems = std_dG_by_system.index
    colors = sns.color_palette("coolwarm", len(systems))
    for color, (sysname, std_dG) in zip(colors, std_dG_by_system.items()):
        plt.bar(sysname, std_dG, edgecolor='black', color=color)
    plt.title(f'LIE STD {component}')
    plt.ylabel('Energy (kcal/mol)');    plt.xlabel('System')
    plt.xticks(rotation=45, ha='right')
    plt.savefig(f'lie_all_STD_{component}.png', dpi=300)
    plt.show()
    plt.close()